## Memorability predictions and length of vector 

In [1]:
# import some libraries first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import mplcyberpunk
from numpy import triu

import gensim
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import gensim.models as models
from scipy.stats import pearsonr

In [2]:
# Insert function first that load csv file
def load_csv_file(filename):
    df = pd.read_csv(filename)
    return df

In [3]:
filepath = '/add/your/path/here'
data_file_path = filepath = '/add/your/path/here'

In [6]:
# pick the dataset 
aka = False
cox = False
madan = False
if aka:
    file = '/AkaMcCoyBhatia_training_dataset_memorabilities.csv'
elif cox:
    file = '/Coxetal_memorability_scores.csv'
elif madan:
    file = '/Madan2021_pRecall_database.csv'
else: 
    file = '/Dymarskyetal_Exp.1-RawMemoryDatawithRCs.csv'

filename = data_file_path + file

In [8]:
data = load_csv_file(filename)
if aka or cox: 
    madan_data = load_csv_file(filepath + '/Madan2021_pRecall_database.csv')
else: 
    print('Madan data already loaded')

Madan data already loaded


In [10]:
# make word all lowercase 
data['Word'] = data['Word'].apply(lambda x: x.lower() if isinstance(x, str) else x)

In [2]:
words = data[['Word']].copy()

## Word2Vec Embeddings 

In [14]:
# Load the pre-trained word2vec-google-news-300 model
model = models.KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

In [15]:
# word embedding function 
def get_word2vec_embedding(word, model):
    try:
        return model[word]
    except KeyError:
        return np.zeros(model.vector_size)  

In [3]:
# Add word embeddings to DataFrame
embedding_size = 300
for i in range(embedding_size):
    words[f'embedding_{i+1}'] = words['Word'].apply(lambda x: get_word2vec_embedding(x, model)[i])

In [18]:
words.to_csv('Aka_word2vec_300dim_embeddings.csv', index=False) 

In [18]:
# Function to compute the length of a vector
def compute_vector_length(embedding_vector):
    return np.linalg.norm(embedding_vector)

In [4]:
# Get the list of embedding column names
embedding_columns = [f'embedding_{i}' for i in range(1, embedding_size + 1)] 

# Add vector lengths to DataFrame
words['vector_length_word2vec'] = words.apply(lambda row: compute_vector_length(row[embedding_columns].values), axis=1)

In [22]:
# Filter out rows with 'vector_length' equal to 0
words_filtered = words[words['vector_length_word2vec'] != 0]
# Create a new DataFrame containing the deleted rows
deleted_rows = words[words['vector_length_word2vec'] == 0]

In [25]:
# create a boolean mask indicating whether each word in 'data' is present in 'deleted_rows'
mask = data['Word'].isin(deleted_rows['Word'])

# filter out rows in 'data' where the word is present in 'deleted_rows'
data_filtered = data[~mask]

In [5]:
data_filtered['vector_length_word2vec'] = words_filtered['vector_length_word2vec']

In [31]:
data_filtered.to_csv('coxetal_words_L2norm.csv', index=False)  

In [42]:
# Calculate Spearman correlation coefficient and p-value for recall
correlation_coefficient_recall, p_value_recall = scipy.stats.spearmanr(data_filtered['vector_length_word2vec'], data_filtered['both_hit'])
print("Pearson correlation coefficient  recognition:", correlation_coefficient_recall)
print("P-value recall:", p_value_recall)
if aka:
    correlation_coefficient_recall, p_value_recall = scipy.stats.spearmanr(words_filtered['vector_length_word2vec'], data_filtered['actual_recall'])
    # Calculate Spearman correlation coefficient and p-value for recognition
    correlation_coefficient_recognition, p_value_recognition = scipy.stats.spearmanr(words_filtered['vector_length_word2vec'], data_filtered['actual_recognition'])
elif cox:
    correlation_coefficient_recall, p_value_recall = scipy.stats.spearmanr(data_filtered['vector_length_word2vec'], data_filtered['recall_memorability'])
    correlation_coefficient_recognition, p_value_recognition= scipy.stats.spearmanr(data_filtered['vector_length_word2vec'], data_filtered['recognition_memorability'])
    print("Pearson correlation coefficient recall:", correlation_coefficient_recall)
    print("P-value recall:", p_value_recall)
    print("Pearson correlation coefficient recognition:", correlation_coefficient_recognition)
    print("P-value recognition:", p_value_recognition)
    
else:
    correlation_coefficient_recall, p_value_recall = scipy.stats.spearmanr(words_filtered['vector_length_word2vec'], data_filtered['pRecall'])
    #correlation_coefficient_recall, p_value_recall = scipy.stats.spearmanr(new_data['vector_length_word2vec'], heard2019_data['pRecall'])

if aka:
    print("Pearson correlation coefficient recognition:", correlation_coefficient_recognition)
    print("P-value recognition:", p_value_recognition)


Pearson correlation coefficient dymarska recognition: 0.47055258790255106
P-value recall: 1.466582875965234e-293


In [8]:
plt.figure(figsize=(8, 6))
sns.jointplot(x=words_filtered['vector_length_word2vec'], y=data_filtered['actual_recognition'], kind='reg', scatter_kws={"alpha": 0.5})
plt.suptitle("Correlation  Vector Length Word2Vec and Recall", y=1.02)
plt.xlabel("Vector Length")
plt.ylabel("Aka Recall memo scores")
plt.show()